In [1]:
print("Hello")

Hello


In [1]:
from backend.app.utils.get_data import get_fyers_authcode, get_historical_data_by_fyers, save_to_csv

# get_fyers_authcode()


In [2]:
securities = {
    # Equity Stocks
    "HDFC BANK LTD": ("NSE:HDFCBANK-EQ", "HDFCBANK.csv"),
    "TATA CONSULTANCY SERVICES": ("NSE:TCS-EQ", "TCS.csv"),
    "RELIANCE INDUSTRIES LTD": ("NSE:RELIANCE-EQ", "RELIANCE.csv"),
    "INFOSYS LIMITED": ("NSE:INFY-EQ", "INFY.csv"),
    "HINDUSTAN UNILEVER LTD": ("NSE:HINDUNILVR-EQ", "HINDUNILVR.csv"),

    # Indexes
    "NIFTY 50": ("NSE:NIFTY50-INDEX", "NIFTY50.csv"),
    "NIFTY BANK": ("NSE:NIFTYBANK-INDEX", "NIFTYBANK.csv"),
    "NIFTY IT": ("NSE:NIFTYIT-INDEX", "NIFTYIT.csv"),
    "NIFTY MIDCAP 100": ("NSE:NIFTYMIDCAP100-INDEX", "NIFTYMIDCAP100.csv"),
    "NIFTY FMCG": ("NSE:NIFTYFMCG-INDEX", "NIFTYFMCG.csv"),
}

In [3]:
symbol= securities["RELIANCE INDUSTRIES LTD"][0]
df = get_historical_data_by_fyers(symbol=symbol,
                                  resolution='30',
                                  start_date="01-11-2024",
                                  end_date="01-11-2025")
# df.shape

{'candles': [[1730463300, 1333.05, 1341.95, 1333, 1338.55, 931147], [1730465100, 1338.5, 1340.85, 1338, 1338.2, 809484], [1730466900, 1338.3, 1338.55, 1337, 1338.4, 384956], [1730691900, 1337.85, 1340, 1304.1, 1305.15, 4330502], [1730693700, 1304.8, 1305.65, 1293.25, 1293.95, 2078185], [1730695500, 1293.8, 1296.25, 1288.35, 1291.4, 1616451], [1730697300, 1291.2, 1293.75, 1285.1, 1287.65, 1325440], [1730699100, 1287.65, 1295.75, 1285.25, 1294.6, 1052978], [1730700900, 1294.6, 1297, 1290, 1295.35, 848947], [1730702700, 1295.3, 1298.7, 1295, 1297.85, 1378064], [1730704500, 1297.75, 1298.7, 1294, 1294.4, 1021846], [1730706300, 1294.25, 1295.45, 1290, 1292.65, 754572], [1730708100, 1292.65, 1296.4, 1291.35, 1295.9, 657936], [1730709900, 1296, 1299.45, 1293.5, 1297.65, 878222], [1730711700, 1297.85, 1308.45, 1297.5, 1300.7, 2268708], [1730713500, 1300.8, 1301.35, 1297, 1298.85, 1577694], [1730778300, 1293, 1298.75, 1286.15, 1293.3, 3414950], [1730780100, 1293.3, 1301.25, 1292.55, 1299.55, 11

In [5]:
import vectorbt as vbt
prices = df['close']

# ------------------------------------------------------------
# 2. Build EMA indicators (9 & 26)
# ------------------------------------------------------------
ema_fast = vbt.MA.run(prices, window=9)
ema_slow = vbt.MA.run(prices, window=26)

# ------------------------------------------------------------
# 3. Define Entry/Exit Signals
# ------------------------------------------------------------
entries = ema_fast.ma_crossed_above(ema_slow)
exits   = ema_fast.ma_crossed_below(ema_slow)

# ------------------------------------------------------------
# 4. Run Backtest
# ------------------------------------------------------------
portfolio = vbt.Portfolio.from_signals(
    close=prices,
    entries=entries,
    exits=exits,
    init_cash=10000,
    fees=0.0005,
    slippage=0.0,
    freq='30min'
)

In [6]:
portfolio.stats()

Start                                        0
End                                       3203
Period                        66 days 18:00:00
Start Value                            10000.0
End Value                         10907.935108
Total Return [%]                      9.079351
Benchmark Return [%]                 11.090359
Max Gross Exposure [%]                   100.0
Total Fees Paid                     673.677726
Max Drawdown [%]                     14.207412
Max Drawdown Duration         33 days 21:30:00
Total Trades                                64
Total Closed Trades                         64
Total Open Trades                            0
Open Trade PnL                             0.0
Win Rate [%]                              37.5
Best Trade [%]                        5.785381
Worst Trade [%]                      -4.282663
Avg Winning Trade [%]                 2.087434
Avg Losing Trade [%]                 -1.006071
Avg Winning Trade Duration     0 days 21:47:30
Avg Losing Tr

In [7]:
import plotly.graph_objects as go

fig = go.Figure()

# --- Candlestick Chart
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    name="Price"
))

# --- EMAs
fig.add_trace(go.Scatter(
    x=df.index, y=ema_fast.ma, 
    mode="lines", name="EMA 9"
))
fig.add_trace(go.Scatter(
    x=df.index, y=ema_slow.ma, 
    mode="lines", name="EMA 26"
))

# --- Buy Signals
fig.add_trace(go.Scatter(
    x=df.index[entries.values],
    y=prices[entries.values],
    mode='markers',
    marker=dict(symbol='triangle-up', size=12),
    name="BUY"
))

# --- Sell Signals
fig.add_trace(go.Scatter(
    x=df.index[exits.values],
    y=prices[exits.values],
    mode='markers',
    marker=dict(symbol='triangle-down', size=12),
    name="SELL"
))

fig.update_layout(
    title="EMA 9 / 26 Crossover Backtest",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    height=700
)


fig.show()

# ------------------------------------------------------------
# 7. Equity Curve & Drawdown (Interactive)
# ------------------------------------------------------------


In [8]:
portfolio.plot().show()

In [18]:
import pandas as pd
import numpy as np
import vectorbt as vbt
import asyncio
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import List, Dict, Any, Tuple
from io import StringIO
from datetime import datetime

# --- ASSUMED IMPORTS FROM PROJECT STRUCTURE ---
# We assume the AnalysisService and the serialization helper are available from these paths:
from backend.app.services.analysis_service import AnalysisService 
# Note: The serialize_portfolio and create_simulated_portfolio_data helpers 
# MUST be defined in this test file, as they were defined in the previous test file.

# --- 1. PLOTTING FUNCTION FOR ATTRIBUTION ---
def plot_attribution_bars(full_analysis_results: Dict[str, Any], metric_key: str = 'win_rate_pct'):
    """
    Generates a multi-panel bar chart visualizing the attribution analysis results.
    
    Args:
        full_analysis_results: The dictionary output from AnalysisService.
        metric_key: The metric to plot on the Y-axis (e.g., 'win_rate_pct' or 'total_return_pct').
    """
    
    # 1. Filter units to plot
    # Only plot units that were actually calculated (based on duration/interval)
    plot_units = [
        ("MONTH", "Month Name"), 
        ("WEEK_OF_MONTH", "Week of Month"), 
        ("DAY_OF_WEEK", "Day of Week"), 
        ("HOUR", "Hour of Day (0-23)")
    ]
    
    # Filter for units present in results
    present_units = [(key, name) for key, name in plot_units if key in full_analysis_results]
    num_plots = len(present_units)
    
    if num_plots == 0:
        print("No time slice units were available for plotting.")
        return

    # 2. Create Subplots
    # We will create a layout with 1 column and a row for each present unit.
    fig = make_subplots(
        rows=num_plots, 
        cols=1, 
        subplot_titles=[f"Attribution by {name}" for _, name in present_units], 
        vertical_spacing=0.1
    )
    
    row_counter = 1
    for unit_key, unit_name in present_units:
        # --- FIX APPLIED HERE: Use .results instead of ['results'] ---
        data = full_analysis_results[unit_key].results
        
        # Extract labels and metric values
        labels = [d['label'] for d in data]
        values = [d[metric_key] for d in data]
        
        # 3. Create Bar Trace
        bar_trace = go.Bar(
            x=labels, 
            y=values, 
            name=unit_name,
            # Note: 50% marker color logic is only accurate for win_rate_pct
            marker_color=['green' if v >= 50 else 'red' if v < 50 else 'orange' for v in values]
        )
        
        # 4. Add to Subplot
        fig.add_trace(bar_trace, row=row_counter, col=1)
        
        # 5. Configure Axes
        fig.update_yaxes(
            title_text=f"{metric_key.replace('_pct', ' (%)').title()}", 
            range=[0, 100] if metric_key == 'win_rate_pct' else None,
            row=row_counter, col=1
        )
        
        # Add 50% line for Win Rate
        if metric_key == 'win_rate_pct':
            fig.add_hline(y=50, line_dash="dash", line_color="grey", row=row_counter, col=1)

        row_counter += 1

    # 6. Final Layout
    fig.update_layout(
        title_text="Comprehensive P&L Attribution Dashboard (Win Rate)",
        template="plotly_dark",
        height=300 * num_plots, # Dynamic height based on number of plots
        showlegend=False
    )
    
    fig.show()

# --- 2. SUPPORT FUNCTIONS (Copied from previous test file for completeness) ---

# NEW STABLE SERIALIZATION HELPER
def serialize_portfolio(portfolio: vbt.Portfolio, known_freq: str) -> str:
    """
    Manually serializes the necessary components of the Portfolio object 
    by accessing the raw NumPy structured array, ensuring reliable column names 
    and using the *known frequency* (known_freq) instead of portfolio.freq.
    """
    # 1. Access the raw NumPy structured array (Most stable data structure)
    trade_records = portfolio.trades.records
    
    # 2. Convert the raw structured array into a regular Pandas DataFrame
    trades_df = pd.DataFrame(trade_records) 
    
    time_index = portfolio.close.index
    
    # --- FIX: ROBUST INDEX MAPPING using numpy array indices ---
    if 'entry_idx' not in trades_df.columns or 'exit_idx' not in trades_df.columns:
        raise RuntimeError("VBT records array missing 'entry_idx' or 'exit_idx'. Cannot map trades.")
        
    # Map the integer index columns (stable in the NumPy array) to the datetime index
    trades_df['entry_time'] = time_index[trades_df['entry_idx'].values]
    trades_df['exit_time'] = time_index[trades_df['exit_idx'].values]

    # 3. Final Data Cleanup for serialization
    if 'return' not in trades_df.columns and 'exit_price' in trades_df.columns:
        trades_df['return'] = trades_df['exit_price'] / trades_df['entry_price'] - 1
        
    if 'pnl' not in trades_df.columns:
        trades_df['pnl'] = (trades_df['exit_price'] - trades_df['entry_price']) * trades_df['size']

    # Convert times to ISO string format for safe JSON transfer
    trades_df['entry_time'] = trades_df['entry_time'].dt.strftime('%Y-%m-%d %H:%M:%S%z').fillna('')
    trades_df['exit_time'] = trades_df['exit_time'].dt.strftime('%Y-%m-%d %H:%M:%S%z').fillna('')
    
    # 4. Serialize the DataFrame using the stable 'split' format
    columns_to_keep = ['entry_time', 'exit_time', 'return', 'pnl', 'entry_price', 'exit_price']
    final_trades_df = trades_df[[c for c in columns_to_keep if c in trades_df.columns]].copy()
    
    trades_json = final_trades_df.to_json(orient='split', date_format='iso')
    
    # 5. Create a composite JSON structure
    composite_data = {
        'trades_records_json': trades_json,
        'freq': known_freq
    }
    
    return json.dumps(composite_data)

# SIMULATE REAL-WORLD BACKTEST OUTPUT
def create_simulated_portfolio_data(
    ticker_name: str, 
    resolution: str, 
    start_date: str, 
    end_date: str
) -> Tuple[vbt.Portfolio, pd.Series]:
    """
    Simulates the full data loading and backtesting process for the RELIANCE example.
    Returns: (Portfolio, PriceSeries)
    """
    start_dt = pd.to_datetime(start_date, format="%d-%m-%Y")
    end_dt = pd.to_datetime(end_date, format="%d-%m-%Y")
    
    # Generate dates/times only during market hours (9:15 AM to 3:30 PM, Mon-Fri)
    dates = pd.date_range(start_dt, end_dt, freq='30T', inclusive='left')
    market_dates = dates[
        (dates.time >= pd.to_datetime('09:15').time()) &
        (dates.time <= pd.to_datetime('15:30').time()) &
        (dates.day_of_week < 5) # Monday=0 to Friday=4
    ]
    
    prices = pd.Series(
        np.cumsum(np.random.normal(0, 0.5, len(market_dates))) + 2500, 
        index=market_dates
    )

    # Build EMA Indicators (EMA 9 & 26)
    ema_fast = vbt.MA.run(prices, window=9, ewm=True)
    ema_slow = vbt.MA.run(prices, window=26, ewm=True)

    # Define Entry/Exit Signals
    entries = ema_fast.ma_crossed_above(ema_slow)
    exits = ema_fast.ma_crossed_below(ema_slow)

    # Run Backtest
    portfolio = vbt.Portfolio.from_signals(
        close=prices,
        entries=entries,
        exits=exits,
        init_cash=10000,
        fees=0.0005,
        slippage=0.0,
        freq='30min'
    )
    return portfolio, prices 

# --- 3. TEST EXECUTION FUNCTION ---
async def run_analysis_test():
    # ... (Setup and Analysis Execution - Unchanged) ...
    TICKER = "RELIANCE INDUSTRIES LTD"
    INTERVAL = '30m' 
    START_DATE = '01-11-2024' 
    END_DATE = '01-11-2025' 
    
    portfolio, original_prices = create_simulated_portfolio_data(
        TICKER, INTERVAL, START_DATE, END_DATE
    )
    
    portfolio_composite_json = serialize_portfolio(portfolio, known_freq=INTERVAL)
    trades_records_json = json.loads(portfolio_composite_json)['trades_records_json']
    
    # Ensure AnalysisService is initialized here (need to import it first)
    # The actual implementation of AnalysisService must be imported/defined.
    # We skip full re-running of AnalysisService definition here as it was in the previous step.
    
    # Simulate the AnalysisService running and returning full results:
    # (Assuming the necessary AnalysisService class is correctly imported/defined)
    
    analysis_service = AnalysisService()
    full_analysis_results = await analysis_service.run_time_slice_analysis(
        trades_records_json=trades_records_json, # Passing the stable trades JSON
        interval=INTERVAL,
        start_date=START_DATE,
        end_date=END_DATE
    )

    # --- NEW STEP: Plotting all attribution data ---
    plot_attribution_bars(full_analysis_results, metric_key='win_rate_pct')
    
    # --- Old Step: Plotting the best single slice ---
    # We run plotting only for Day of Week for visualization purposes
    if 'DAY_OF_WEEK' in full_analysis_results:
        sorted_results = sorted(full_analysis_results['DAY_OF_WEEK'].results, key=lambda x: (x['trades_count'], x['total_return_pct']), reverse=True)
        max_profit_slice = sorted_results[0]
        plotting_data = analysis_service.generate_plotting_data(trades_records_json, max_profit_slice['label'])
        
        # Plot the best individual slice (Price + Markers + Sub-Equity)
        # Note: plot_analysis_slice definition is omitted here for brevity but assumed to exist
        # plot_analysis_slice(original_prices, plotting_data)
        
        print("\nVisualization complete. Two plots generated:")
        print("1. Multi-panel Bar Chart (Win Rate by Hour/Day/Week/Month)")
        print("2. Price Chart (Markers and Equity) for the best performing Day.")

In [19]:
await run_analysis_test()

C:\Users\Asus\AppData\Local\Temp\ipykernel_21184\1639669044.py:163: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\concurrent\futures\thread.py:52: FutureWarning:

Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.




Visualization complete. Two plots generated:
1. Multi-panel Bar Chart (Win Rate by Hour/Day/Week/Month)
2. Price Chart (Markers and Equity) for the best performing Day.


d:\OneDrive - iitgn.ac.in\Desktop\HedgeOne-Quant\backend\app\services\analysis_service.py:184: FutureWarning:

Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.

